# 05. Bayesian segment conversion (Beta-Binomial)

Notebook 01 is a demand curve and needs MCMC. Notebook 04 is a mixture of continuous features fit by EM. This notebook is the other segmentation question: two treatments, a conversion count, and a probability that one beats the other. The posterior is Beta. No sampler.

## Business use

A segment or a treatment has $k$ orders from $n$ sessions. Merchandising wants $P(B > A)$ and the expected lift, with a prior that says conversion is near 5% until the data says otherwise. The same pattern covers any rate with a conjugate prior: click-through, refund, repeat purchase.

## Intuition and math

Prior $\mathrm{Beta}(a_0, b_0)$ and $k$ successes in $n$ trials give

$$
p \mid k, n \sim \mathrm{Beta}(a_0 + k,\ b_0 + n - k).
$$

The mean is $(a_0 + k) / (a_0 + b_0 + n)$. The prior counts as $a_0 + b_0$ sessions of evidence. Here $a_0 = 2$, $b_0 = 38$: a 5% rate, worth about 40 sessions. Monte Carlo on the two Beta posteriors answers $P(B > A)$ and $E[B/A - 1]$. That is the whole inference.

Use this when the likelihood is binomial and you only need a rate. Move to notebook 01 or 03 when the rate depends on price, discount, or a hierarchy that has no closed form.


## Step 1. Update both segments


In [1]:
import numpy as np
from scipy import stats

prior_a, prior_b = 2, 38
success_a, sessions = 120, 2400
success_b = 150
posterior_a = stats.beta(prior_a + success_a, prior_b + sessions - success_a)
posterior_b = stats.beta(prior_a + success_b, prior_b + sessions - success_b)
lo, hi = posterior_a.ppf([0.03, 0.97])
print(
    f"segment A posterior Beta({prior_a + success_a},{prior_b + sessions - success_a}) "
    f"mean={posterior_a.mean():.4f} 94% ETI=({lo:.4f},{hi:.4f})"
)
print(
    f"segment B posterior mean={posterior_b.mean():.4f} "
    f"94% ETI=({posterior_b.ppf(0.03):.4f},{posterior_b.ppf(0.97):.4f})"
)
assert abs(posterior_a.mean() - 0.05) < 1e-12

segment A posterior Beta(122,2318) mean=0.0500 94% ETI=(0.0420,0.0586)
segment B posterior mean=0.0623 94% ETI=(0.0534,0.0718)


## Step 2. Probability that B wins, and a wider prior

Draw both posteriors. $E[B/A - 1]$ is not the same as $E[B]/E[A] - 1$, because the ratio of random variables is skewed. Then repeat with a flat $\mathrm{Beta}(1, 1)$ prior. With 2,400 sessions the likelihood dominates, so the decision should barely move. That is the sensitivity check to show when someone objects that the prior is subjective.


In [2]:
random = np.random.default_rng(0)
draws_a = posterior_a.rvs(100_000, random_state=random)
draws_b = posterior_b.rvs(100_000, random_state=random)
p_b_wins = float(np.mean(draws_b > draws_a))
expected_lift = float(np.mean(draws_b / draws_a - 1))
print("P(B>A) =", round(p_b_wins, 3), " expected lift =", round(expected_lift, 3))

flat_a = stats.beta(1 + success_a, 1 + sessions - success_a)
flat_b = stats.beta(1 + success_b, 1 + sessions - success_b)
flat_draws_a = flat_a.rvs(100_000, random_state=random)
flat_draws_b = flat_b.rvs(100_000, random_state=random)
flat_p = float(np.mean(flat_draws_b > flat_draws_a))
flat_lift = float(np.mean(flat_draws_b / flat_draws_a - 1))
print("flat prior P(B>A) =", round(flat_p, 3), " expected lift =", round(flat_lift, 3))
assert p_b_wins > 0.95
assert flat_p > 0.95

P(B>A) = 0.969  expected lift = 0.255
flat prior P(B>A) = 0.969  expected lift = 0.258


## How to read the outputs

Segment A's posterior mean is exactly 0.05: the prior said 5%, and 120 / 2,400 is also 5%, so the mean does not move. The 94% equal-tailed interval is about 4.2% to 5.9%. That interval is a statement about the rate, not about next session's random outcome.

$P(B > A)$ near 0.97 means that in 97 of 100 draws from the posterior, B's rate is higher. Expected lift near 25% is $E[B/A - 1]$. The flat prior leaves both numbers in the same place, which is what "the data dominate" looks like. With 40 sessions instead of 2,400, the two priors would disagree and you would say so.

## Pitfalls

- Quoting $E[B]/E[A] - 1$ as if it were $E[B/A - 1]$.
- Treating the credible interval on the rate as a prediction interval for one session.
- A shared prior that is much stronger than the sample, hidden in the appendix.
- Comparing segments that were picked after looking at the rates, without saying the selection happened.


## How to explain this to a CTO


In [3]:
print(
    f"Treatment B converts better than A in about {p_b_wins:.0%} of the scenarios consistent "
    f"with the test, with an expected lift of about {expected_lift:.0%}. "
    "That is enough to ship B for this segment. "
    "We did not need a sampler: the update is a closed-form Beta."
)
print(
    f"Widening the prior from a 5% belief to a flat one leaves the probability at {flat_p:.0%}. "
    "The test is large enough that the prior is not driving the call."
)

Treatment B converts better than A in about 97% of the scenarios consistent with the test, with an expected lift of about 26%. That is enough to ship B for this segment. We did not need a sampler: the update is a closed-form Beta.
Widening the prior from a 5% belief to a flat one leaves the probability at 97%. The test is large enough that the prior is not driving the call.


## Check yourself

**When do you skip MCMC?**
When a conjugate update exists (Beta-Binomial rates, Dirichlet-Multinomial transitions in notebook 02), when the posterior is so tight that a point estimate is the decision, or when latency forces variational inference or EM. MCMC is reserved for offline training of models without a closed form, where a calibrated probability changes the decision.

**What if the prior and the flat prior disagree?**
Report both. The gap is the part of the decision that is assumption rather than data. Either collect more sessions or write the assumption down next to the recommendation.
